## Master’s Thesis – Transkribus Export

This notebook exports already digitized magazine issues
from Transkribus using ACDH tooling.

[Documentation and where I got the code from](https://github.com/acdh-oeaw/acdh-transkribus-utils)

I used the acdh-transkribus-utils Python library (MIT License) to download and export XML from the Transkribus REST API (see https://github.com/acdh-oeaw/acdh-transkribus-utils). The original MIT license and copyright notice are preserved.



Author: Sophie Hamann
Created: 2026-01-20

In [25]:
# defining 
from pathlib import Path

DATA_BASE = Path.home() / "Documents" / "MA_Universität_Wien" / "master_thesis" / "master_thesis_data"

RAW_DIR = DATA_BASE / "raw"
PROCESSED_DIR = DATA_BASE / "processed"

RAW_TRANSKRIBUS_XML = RAW_DIR / "transkribus" / "xml"
RAW_PAGE_XML = RAW_DIR / "transkribus" / "page_xml"
RAW_PDF = RAW_DIR / "pdf" / "issues"

PROCESSED_TEI = PROCESSED_DIR / "tei"
PROCESSED_GT = PROCESSED_DIR / "training_data" / "ground_truth"

In [26]:
# make sure this directory structure exists
for d in [
    RAW_TRANSKRIBUS_XML,
    RAW_PAGE_XML,
    PROCESSED_TEI,
    PROCESSED_GT,
]:
    d.mkdir(parents=True, exist_ok=True)

In [27]:
import os

from transkribus_utils.transkribus_utils import ACDHTranskribusUtils


tr_user = os.environ.get("TRANSKRIBUS_USER")
tr_pw = os.environ.get("TRANSKRIBUS_PASSWORD")

client = ACDHTranskribusUtils(user=tr_user, password=tr_pw)

# List all Collections

In [28]:
collections = client.list_collections()
for x in collections[-7:]:
    print(x["colId"], x["colName"])

2189341 Quick Text Recognition
2204941 heresies_collection
2222244 heresies_training_dataset
2257154 Digitizing_Heresies
2291396 baseline_training_data
2297059 baseline_test
2324844 digitizing hard stuff


In [29]:
# inspecting 2204941 heresies_collection 
type (documents)
len(documents)

# sorting documents by title
documents = sorted(documents, key=lambda d: d["title"])
[d["title"] for d in documents]

['heresies_01',
 'heresies_02',
 'heresies_03',
 'heresies_04',
 'heresies_05',
 'heresies_06',
 'heresies_07',
 'heresies_08',
 'heresies_09',
 'heresies_10',
 'heresies_11',
 'heresies_12',
 'heresies_13',
 'heresies_14',
 'heresies_15',
 'heresies_16',
 'heresies_17',
 'heresies_18',
 'heresies_19',
 'heresies_20',
 'heresies_21',
 'heresies_22',
 'heresies_23',
 'heresies_24',
 'heresies_25',
 'heresies_26',
 'heresies_27']

In [34]:
# creating a look up to find the corresponding docId for a given document title
# “Document identifiers were retrieved programmatically via the Transkribus REST API and stored in a lookup structure 
# to ensure reproducible access and avoid hard-coded identifiers.” (chat gpt)
doc_map = {d["title"]: d["docId"] for d in documents}
doc_map["heresies_01"]

11509115

In [35]:
# defining working collection ID
col_id = 2204941          # heresies collection
documents = client.list_docs(col_id) # get all documents in collection

https://transkribus.eu/TrpServer/rest/collections/2204941/list


# List all documents from the given collection (heresies_collection)

In [36]:
# checking what metadata fields are available in a collection
for k in documents[-3:]:
    print(k.keys())

dict_keys(['type', 'docId', 'title', 'uploadTimestamp', 'uploader', 'uploaderId', 'nrOfPages', 'pageId', 'url', 'thumbUrl', 'status', 'fimgStoreColl', 'origDocId', 'collectionList', 'attributes', 'labels', 'pageLabels', 'mainColId'])
dict_keys(['type', 'docId', 'title', 'uploadTimestamp', 'uploader', 'uploaderId', 'nrOfPages', 'pageId', 'url', 'thumbUrl', 'status', 'fimgStoreColl', 'origDocId', 'collectionList', 'attributes', 'labels', 'pageLabels', 'mainColId'])
dict_keys(['type', 'docId', 'title', 'uploadTimestamp', 'uploader', 'uploaderId', 'nrOfPages', 'pageId', 'url', 'thumbUrl', 'status', 'fimgStoreColl', 'origDocId', 'collectionList', 'attributes', 'labels', 'pageLabels', 'mainColId'])


In [39]:
# exploratory inspection of document metadata
documents = client.list_docs(col_id)

for doc in documents:
    print(
        doc["docId"],
        doc["title"],
        doc["nrOfPages"]
    )

https://transkribus.eu/TrpServer/rest/collections/2204941/list
11501518 heresies_02 128
11501519 heresies_03 122
11501520 heresies_05 139
11501521 heresies_07 130
11501522 heresies_09 100
11501523 heresies_13 100
11501559 heresies_04 132
11501560 heresies_06 132
11501561 heresies_08 132
11501562 heresies_10 100
11501563 heresies_11 98
11501564 heresies_12 100
11501565 heresies_14 42
11501566 heresies_15 76
11501567 heresies_16 98
11501568 heresies_17 100
11501569 heresies_18 97
11501570 heresies_19 33
11501571 heresies_20 100
11501573 heresies_24 42
11501574 heresies_25 100
11501775 heresies_21 84
11501777 heresies_22 100
11501779 heresies_23 100
11501781 heresies_26 112
11501783 heresies_27 121
11509115 heresies_01 116


# Inspect one Document from the Given Collection
This part of the code inspects a single document out of the Transkribus heresies_collection.
I am looking at document metadata, page lists, page xml and transcripts.

All Python code used in this thesis for accessing, inspecting, and exporting Transkribus documents is based exclusively on the publicly available and MIT-licensed acdh-transkribus-utils library, without introducing additional custom functionality beyond the methods provided in the original repository.
[see here](https://github.com/acdh-oeaw/acdh-transkribus-utils/blob/main/transkribus_utils/transkribus_utils.py)

In [37]:
# defining document ID
doc_id_1 = 11509115         # heresies_01

In [38]:
# inspecting document metadata
doc_metadata_1 = client.get_doc_md(doc_id_1, col_id)
print(doc_metadata_1)    

{'type': 'trpDocMetadata', 'docId': 11509115, 'title': 'heresies_01', 'uploadTimestamp': 1760968484613, 'uploader': 'a12248219@unet.univie.ac.at', 'uploaderId': 447621, 'nrOfPages': 116, 'pageId': 104610900, 'url': 'https://files.transkribus.eu/Get?fileType=view&id=NWEWFYMAIPUDDPYUUBMMLBCD', 'thumbUrl': 'https://files.transkribus.eu/Get?fileType=thumb&id=NWEWFYMAIPUDDPYUUBMMLBCD', 'status': 0, 'fimgStoreColl': 'TrpDoc_DEA_11509115', 'origDocId': 0, 'collectionList': {'colList': [{'colId': 2204941, 'colName': 'heresies_collection', 'description': 'created by a12248219@unet.univie.ac.at', 'crowdsourcing': False, 'elearning': False, 'nrOfDocuments': 0, 'isFavorite': False}]}, 'attributes': [], 'labels': [], 'pageLabels': [], 'mainColId': 2204941}


In [ ]:
# inspect document overview in markdown format
overview = client.get_doc_overview_md(doc_id_1, col_id_1)
print(overview["trp_return"]["md"])

In [ ]:
#inspecting full xml
fulldoc = client.get_fulldoc_md(doc_id_1, col_id_1, page_id=5)
print(fulldoc["doc_xml"])

In [ ]:
fulldoc = client.get_transcript(fulldoc)
print(fulldoc["transcript"][:10])

In [ ]:
# I want to see the xml structure of a page
from lxml import etree

print(etree.tostring(fulldoc["doc_xml"], pretty_print=True).decode())

# This is just checking whether this xml file has something in it. 
This code has been generated by Chat GPT

In [ ]:
fulldoc = client.get_transcript(fulldoc)
page_xml = fulldoc["page_xml"]
ns = {"page": "http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15"}
len(page_xml.xpath(".//page:TextLine", namespaces=ns))

In [ ]:
chars = page_xml.xpath(".//page:TextLine//page:Unicode/text()", namespaces=ns)
sum(len(c) for c in chars)

# Download METS files from Collection
The METS files exported from Transkribus serve as structural containers and reference PAGE-XML files, in which the actual transcribed text is encoded within <TextLine> and <Unicode> elements.

In [40]:
from transkribus_utils.transkribus_utils import ACDHTranskribusUtils

COL_ID = 2204941          # heresies collection

client.collection_to_mets(
    COL_ID,
    file_path=str(RAW_TRANSKRIBUS_XML)
)

https://transkribus.eu/TrpServer/rest/collections/2204941/list
27 to download


KeyboardInterrupt: 

This code exports the METS XML for a single specified document from a Transkribus collection and stores it locally.

In [ ]:
COL_ID = 2204941          # heresies collection
DOC_ID = 11509115  

client = ACDHTranskribusUtils()

client.collection_to_mets(
    COL_ID,
    file_path="./foo",
    filter_by_doc_ids=[DOC_ID]
)

# Download PAGE-XML with the actual text (issue 1)
# DONE WITH CHAT GPT (cite correctly!)

In [ ]:
from transkribus_utils.transkribus_utils import ACDHTranskribusUtils
from lxml import etree
import os

client = ACDHTranskribusUtils()

COL_ID = 2204941
DOC_ID = 11509115  # heresies_01

In [ ]:
overview = client.get_doc_overview_md(DOC_ID, COL_ID)
pages = overview["pages"]

In [ ]:
# one file per page
os.makedirs("heresies_01_pagexml", exist_ok=True)

for p in pages:
    page_id = p["page_nr"]

    fulldoc = client.get_fulldoc_md(DOC_ID, COL_ID, page_id=page_id)
    fulldoc = client.get_transcript(fulldoc)

    page_xml = fulldoc["page_xml"]

    out_file = f"heresies_01_pagexml/page_{page_id}.xml"
    with open(out_file, "wb") as f:
        f.write(etree.tostring(page_xml, pretty_print=True))

In [ ]:
# one file per issue
from lxml import etree
import os

root = etree.Element("Document")

for p in pages:
    page_id = p["page_nr"]

    fulldoc = client.get_fulldoc_md(DOC_ID, COL_ID, page_id=page_id)
    fulldoc = client.get_transcript(fulldoc)

    page_xml = fulldoc["page_xml"]

    # import PAGE-XML under a single root
    root.append(page_xml)

# write one combined file
with open("heresies_01_pagexml.xml", "wb") as f:
    f.write(etree.tostring(root, pretty_print=True))